# 7 Synthesis: Double Compression

Display-only synthesis for the diversity-facet outputs from notebooks 5a/5b/6a/6b.

Hard rule: this notebook does not recompute embeddings, permutations, review panels, or any M1-M5 metric. It only reads tidy result tables, computes normalized AI/Human ratios where valid, and exports figures/tables for manuscript synthesis.


## Configuration

In [ ]:
import os
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', str(Path('/tmp') / 'codex-matplotlib'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from proposal_generation import find_project_root

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
CONDITIONS = ['baseline', 'one_at_a_time', 'persona']
BRANCH = 'rephrased'
PRIMARY_METRIC = 'vendi'
PRIMARY_PARAM = 'q=1'
MODELS = ['human_vs_claude', 'human_vs_gemini', 'human_vs_gpt', 'human_vs_pooled_ai']
MODEL_LABELS = {
    'human_vs_claude': 'Claude',
    'human_vs_gemini': 'Gemini',
    'human_vs_gpt': 'GPT',
    'human_vs_pooled_ai': 'All AI',
}
PALETTE = {'Human': '#DC143C', 'Claude': '#4A90E2', 'Gemini': '#7B68EE', 'GPT': '#3CB371', 'All AI': '#4A90E2'}
CONDITION_MARKERS = {'baseline': 'o', 'one_at_a_time': 's', 'persona': '^'}

sns.set_theme(style='whitegrid', context='talk')
TABLE_OUT = PROJECT_ROOT / 'results' / 'tables' / 'synthesis' / BRANCH
FIG_OUT = PROJECT_ROOT / 'results' / 'figures' / 'synthesis' / BRANCH
TABLE_OUT.mkdir(parents=True, exist_ok=True)
FIG_OUT.mkdir(parents=True, exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')


## Load Tidy Result Tables

In [ ]:
def facet_path(condition, task, branch=BRANCH):
    return PROJECT_ROOT / 'results' / 'tables' / condition / task / branch / 'facet' / 'facet_diversity_tests.csv'

def gradient_path(condition, task, branch=BRANCH):
    return PROJECT_ROOT / 'results' / 'tables' / condition / task / branch / 'facet' / 'facet_diversity_gradient.csv'

frames = []
missing = []
for condition in CONDITIONS:
    for task in ['proposals', 'reviews']:
        path = facet_path(condition, task)
        if not path.exists():
            missing.append(path)
            continue
        df = pd.read_csv(path)
        if 'task' not in df.columns:
            df['task'] = task
        if 'text_branch' not in df.columns:
            df['text_branch'] = BRANCH
        frames.append(df)
if missing:
    raise FileNotFoundError('Run 5b/6b facet sections first; missing:' + chr(10) + chr(10).join(str(path) for path in missing))
T = pd.concat(frames, ignore_index=True)
T = T[T['text_branch'].eq(BRANCH)].copy()
print(T.shape)
display(T.head())


## Compute Ratios

In [ ]:
T['model_label'] = T['comparison'].map(MODEL_LABELS).fillna(T['comparison'])
T['parity_ref'] = 1.0
coverage_mask = T['metric'].eq('coverage_geometric')
T.loc[coverage_mask, 'parity_ref'] = pd.to_numeric(T.loc[coverage_mask, 'human_value'], errors='coerce')

is_ratio_metric = ~T['facet'].eq('displacement')
T['ratio'] = np.nan
regular_mask = is_ratio_metric & ~coverage_mask
T.loc[regular_mask, 'ratio'] = pd.to_numeric(T.loc[regular_mask, 'ai_value'], errors='coerce') / pd.to_numeric(T.loc[regular_mask, 'human_value'], errors='coerce')
T.loc[coverage_mask, 'ratio'] = pd.to_numeric(T.loc[coverage_mask, 'ai_value'], errors='coerce') / T.loc[coverage_mask, 'parity_ref']
T['log2ratio'] = np.log2(T['ratio'])
axis_mode = 'log2ratio' if (T.loc[is_ratio_metric, 'ratio'] > 1).any() else 'ratio'
print('Axis mode:', axis_mode)
T.to_csv(TABLE_OUT / 'double_compression_summary.csv', index=False)
display(T[['condition','task','field','comparison','facet','metric','param','human_value','ai_value','ratio','p_fdr']].head(20))


## Figure 1 Slopegraph

In [ ]:
def sig_star(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return ''

def primary_rows(metric=PRIMARY_METRIC, param=PRIMARY_PARAM, field='whole'):
    return T[(T['metric'].eq(metric)) & (T['param'].fillna('').eq(param)) & (T['field'].eq(field)) & (T['comparison'].isin(MODELS))].copy()

def slopegraph(metric=PRIMARY_METRIC, param=PRIMARY_PARAM, branch=BRANCH):
    df = primary_rows(metric, param, field='whole')
    fig, axes = plt.subplots(1, len(CONDITIONS), figsize=(5.2 * len(CONDITIONS), 5), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, condition in zip(axes, CONDITIONS):
        sub = df[df['condition'].eq(condition)].copy()
        for comp in MODELS:
            gen = sub[(sub['task'].eq('proposals')) & (sub['comparison'].eq(comp))]
            filt = sub[(sub['task'].eq('reviews')) & (sub['comparison'].eq(comp))]
            if gen.empty or filt.empty:
                continue
            y = [float(gen['ratio'].iloc[0]), float(filt['ratio'].iloc[0])]
            label = MODEL_LABELS[comp]
            ax.plot([0, 1], y, color=PALETTE.get(label, '#808080'), marker='o', linewidth=2, label=label)
            for x, row in zip([0, 1], [gen.iloc[0], filt.iloc[0]]):
                star = sig_star(row.get('p_fdr'))
                if star:
                    ax.text(x, row['ratio'], star, ha='center', va='bottom', fontsize=10)
        ax.axhline(1.0, color='black', linewidth=1, linestyle='--')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Generation', 'Filtering'])
        ax.set_title(condition)
        ax.set_ylabel('Diversity retained (AI / Human parity)')
        ax.set_ylim(bottom=0)
    handles, labels = axes[-1].get_legend_handles_labels()
    if handles:
        axes[-1].legend(handles, labels, loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=False)
    fig.suptitle(f'Double-compression slopegraph: {metric} {param}', y=1.03)
    fig.tight_layout()
    fig.savefig(FIG_OUT / 'fig1_double_compression_slopegraph.png', dpi=220, bbox_inches='tight')
    fig.savefig(FIG_OUT / 'fig1_double_compression_slopegraph.pdf', bbox_inches='tight')
    return fig

fig = slopegraph()
plt.show()


## Figure 2 Compression Map

In [ ]:
def compression_map(metric=PRIMARY_METRIC, param=PRIMARY_PARAM):
    df = primary_rows(metric, param, field='whole')
    gen = df[df['task'].eq('proposals')][['condition','comparison','model_label','ratio','parity_ref','p_fdr']].rename(columns={'ratio':'ratio_gen','parity_ref':'parity_gen','p_fdr':'p_fdr_gen'})
    filt = df[df['task'].eq('reviews')][['condition','comparison','ratio','parity_ref','p_fdr']].rename(columns={'ratio':'ratio_filt','parity_ref':'parity_filt','p_fdr':'p_fdr_filt'})
    P = gen.merge(filt, on=['condition','comparison'], how='inner')
    fig, ax = plt.subplots(figsize=(7, 6))
    for model, model_df in P.groupby('model_label'):
        model_df = model_df.set_index('condition').reindex(CONDITIONS).dropna(subset=['ratio_gen','ratio_filt'])
        ax.plot(model_df['ratio_gen'], model_df['ratio_filt'], color=PALETTE.get(model, '#808080'), alpha=0.45, linewidth=1.5)
        for condition, row in model_df.iterrows():
            ax.scatter(row['ratio_gen'], row['ratio_filt'], color=PALETTE.get(model, '#808080'), marker=CONDITION_MARKERS.get(condition, 'o'), s=90, edgecolor='black', linewidth=0.6, label=f'{model} {condition}')
    ax.axvline(1.0, color='black', linestyle='--', linewidth=1)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_xlabel('Generation diversity retained')
    ax.set_ylabel('Filtering diversity retained')
    ax.set_title(f'Compression map: {metric} {param}')
    handles, labels = ax.get_legend_handles_labels()
    dedup = dict(zip(labels, handles))
    ax.legend(dedup.values(), dedup.keys(), loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=False, fontsize=8)
    fig.tight_layout()
    fig.savefig(FIG_OUT / 'fig2_compression_map.png', dpi=220, bbox_inches='tight')
    fig.savefig(FIG_OUT / 'fig2_compression_map.pdf', bbox_inches='tight')
    P.to_csv(TABLE_OUT / 'compression_map_points.csv', index=False)
    return fig, P

fig, compression_points = compression_map()
plt.show()
display(compression_points)


## Figure 3 Robustness Grid

In [ ]:
ROBUSTNESS_SPECS = [
    ('vendi', 'q=1'),
    ('coverage_geometric', 'k=3'),
    ('participation_ratio', ''),
    ('ripley_excess', 'r=pooled_q01_q50'),
]

def robustness_grid():
    fig, axes = plt.subplots(len(ROBUSTNESS_SPECS), 2, figsize=(13, 4 * len(ROBUSTNESS_SPECS)), sharey='row')
    for r, (metric, param) in enumerate(ROBUSTNESS_SPECS):
        for c, task in enumerate(['proposals', 'reviews']):
            ax = axes[r, c]
            if metric == 'coverage_geometric':
                param_mask = T['param'].fillna('').str.startswith('k=')
            else:
                param_mask = T['param'].fillna('').eq(param)
            sub = T[(T['task'].eq(task)) & (T['field'].eq('whole')) & (T['metric'].eq(metric)) & param_mask & (T['comparison'].isin(MODELS))].copy()
            if sub.empty:
                ax.axis('off')
                continue
            sns.barplot(data=sub, x='condition', y='ratio', hue='model_label', palette=PALETTE, ax=ax)
            ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
            ax.set_title(f'{task}: {metric} {param}')
            ax.set_xlabel('')
            ax.set_ylabel('AI / Human parity')
            if r != 0 or c != 1:
                ax.legend_.remove() if ax.legend_ else None
            else:
                ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=False)
    fig.suptitle('Facet robustness grid', y=1.0)
    fig.tight_layout()
    fig.savefig(FIG_OUT / 'fig3_robustness_grid.png', dpi=220, bbox_inches='tight')
    fig.savefig(FIG_OUT / 'fig3_robustness_grid.pdf', bbox_inches='tight')
    return fig

fig = robustness_grid()
plt.show()


## Figure 4 Paired UMAPs (Illustration Only)

In [ ]:
def load_umap_frame(condition, task, branch=BRANCH):
    root = PROJECT_ROOT / 'data' / 'prepared' / condition / task / branch
    if task == 'proposals':
        master = pd.read_csv(root / 'proposal_master.csv')
        coords = np.load(root / 'proposal_umap2d.npy')
        group = np.where(master['source_type'].eq('human'), 'Human', master['source_group'].astype(str))
        return pd.DataFrame({'task': task, 'group': group, 'x': coords[:,0], 'y': coords[:,1]})
    master = pd.read_csv(root / 'review_master.csv')
    coords = np.load(root / 'review_umap2d.npy')
    group = np.where(master['review_source'].eq('human'), 'Human', master['source_family'].str.title())
    return pd.DataFrame({'task': task, 'group': group, 'x': coords[:,0], 'y': coords[:,1]})

def paired_umaps(condition='baseline'):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, task in zip(axes, ['proposals', 'reviews']):
        df = load_umap_frame(condition, task)
        sns.scatterplot(data=df, x='x', y='y', hue='group', palette=PALETTE, s=35, alpha=0.75, ax=ax)
        ax.set_title(f'{condition}: {task} UMAP illustration')
        ax.set_xlabel('UMAP 1')
        ax.set_ylabel('UMAP 2')
        ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), frameon=False, fontsize=8)
    fig.suptitle('Illustration only: all metrics above are computed in full embedding space', y=1.03)
    fig.tight_layout()
    fig.savefig(FIG_OUT / f'fig4_paired_umaps_{condition}.png', dpi=220, bbox_inches='tight')
    fig.savefig(FIG_OUT / f'fig4_paired_umaps_{condition}.pdf', bbox_inches='tight')
    return fig

for condition in CONDITIONS:
    fig = paired_umaps(condition)
    plt.show()


## Figure 5 Gradient Panel

In [ ]:
def gradient_panel(metric=PRIMARY_METRIC, param=PRIMARY_PARAM):
    sub = T[(T['comparison'].eq('human_vs_pooled_ai')) & (T['field'].eq('whole')) & (T['metric'].eq(metric)) & (T['param'].fillna('').eq(param))].copy()
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.lineplot(data=sub, x='condition', y='ratio', hue='task', marker='o', linewidth=2.2, ax=ax)
    ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
    ax.set_title(f'Pooled All-AI retained diversity by condition: {metric} {param}')
    ax.set_ylabel('AI / Human parity')
    ax.set_xlabel('')
    fig.tight_layout()
    fig.savefig(FIG_OUT / 'fig5_cross_condition_gradient_panel.png', dpi=220, bbox_inches='tight')
    fig.savefig(FIG_OUT / 'fig5_cross_condition_gradient_panel.pdf', bbox_inches='tight')
    return fig

fig = gradient_panel()
plt.show()
print('Saved summary table to:', TABLE_OUT / 'double_compression_summary.csv')
print('Saved figures to:', FIG_OUT)
